# Фінальний проєкт. FitCoach – фітнес-асистент
**Курс:** AI Fundamentals · Product Management · Катерина Карпова

**Для кого:** жінки 40–45 років із сидячою роботою, яких турбує біль у спині чи шиї і які бояться собі нашкодити.

**Що вміє:** рахує ІМТ і норму калорій, підбирає безпечні вправи для дому й складає план тренування на потрібну кількість хвилин.

**Як влаштовано:** граф LangGraph. Дані профілю й тривожні ознаки розбирає код, модель лише обирає інструмент і формулює відповідь.

**Основа:** ноутбук теми 8 (ДЗ4). Попередня версія збережена в історії репозиторію.

⚡ **Запуск:** ключ `OPENAI_API_KEY` у Colab Secrets (або у файлі `.env` локально), далі комірки зверху вниз.

## 0. Налаштування

In [1]:
# Встановлення бібліотек потрібне лише в Google Colab: локально вони вже є у віртуальному середовищі.
# Версії зафіксовані (ті самі, що в requirements.txt), щоб ноутбук поводився однаково всюди.
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "langchain==1.3.11", "langchain-openai==1.3.3", "langgraph==1.2.7",
         "python-dotenv==1.2.2", "gradio==6.32.0"],
        check=True,
    )
    print("✅ Бібліотеки встановлено в Colab")
else:
    print("✅ Локальний запуск: бібліотеки з віртуального середовища")

✅ Локальний запуск: бібліотеки з віртуального середовища


In [2]:
import os
import getpass

from langchain_openai import ChatOpenAI

# Ключ: у Colab з Secrets, локально з файлу .env; якщо ніде немає, попросимо ввести (ключ не виводиться)
if IN_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv(".env")

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

# Та сама модель і параметри, що в ДЗ4: так різницю в поведінці можна приписати архітектурі, а не моделі
MODEL_NAME = "gpt-4o-mini"
chat_model = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0.3,
    max_tokens=512,
    timeout=30,
)

check = chat_model.invoke("Одним коротким реченням підтвердь, що ти готовий працювати.")
print(f"✅ Модель {MODEL_NAME} відповідає: {check.content}")

✅ Модель gpt-4o-mini відповідає: Я готовий працювати!


## 1. Дані

База вправ для дому й рівні активності. Протипоказання розмічені за загальнодоступними рекомендаціями; перед продакшном їх має перевірити фізіотерапевт.

In [4]:
# База вправ для дому. Ключі груп англійською, "label" — назва для людини,
# "aliases" — основи слів, за якими код упізнає групу в будь-якому відмінку ("спин" → "спина", "спину").
# minutes — орієнтовна тривалість вправи разом з відпочинком (для плану тренування).
# contraindications — стани, за яких вправу не пропонуємо:
#   lower_back_pain — біль у попереку, neck_pain — біль у шиї, acute_back_injury — гостра травма спини.
# Розминка й заминка — окремі групи: їх бере план тренування для початку й кінця.

def ex(name, level, technique, sets, reps, minutes, contraindications=()):
    return {"name": name, "level": level, "technique": technique, "sets": sets, "reps": reps,
            "minutes": minutes, "contraindications": list(contraindications)}

T, P, S = "acute_back_injury", "lower_back_pain", "neck_pain"

EXERCISES_DB = {
    "warmup": {"label": "розминка", "aliases": ["розминк", "розігрів"], "exercises": [
        ex("Ходьба на місці", "beginner", "Крокуй на місці в комфортному темпі, руки рухаються вільно", "1", "2 хвилини", 2),
        ex("Кругові рухи плечима", "beginner", "Повільно обертай плечима назад, потім уперед", "1", "10 у кожен бік", 1),
        ex("Махи руками перед грудьми", "beginner", "Розведи руки в сторони й плавно схрещуй їх перед собою", "1", "15", 1),
        ex("Підйоми на носки", "beginner", "Стоячи біля опори, плавно підіймайся на носки й опускайся", "1", "15", 1),
        ex("Кругові рухи тазом", "beginner", "Руки на поясі, повільно описуй тазом невеликі кола", "1", "8 у кожен бік", 1, [T]),
        ex("М'які повороти тулуба стоячи", "beginner", "Стопи на ширині плечей, плавно повертай корпус вправо й уліво без ривків", "1", "10 у кожен бік", 1, [T]),
    ]},
    "back": {"label": "спина", "aliases": ["спин", "попере"], "exercises": [
        ex("Котик-корова", "beginner", "Стоячи на четвереньках, по черзі плавно прогинай і округлюй спину", "2", "10", 2, [T]),
        ex("Нахили таза лежачи", "beginner", "Лежачи на спині, ноги зігнуті, м'яко притискай поперек до підлоги й розслабляй", "2", "10", 2, [T]),
        ex("Сідничний міст", "beginner", "Лежачи на спині, ноги зігнуті, плавно підіймай таз і опускай", "2-3", "10-12", 3, [T]),
        ex("Птах-собака", "beginner", "На четвереньках витягни протилежні руку й ногу, затримайся на 3 секунди, поміняй сторону", "2", "8 на кожну сторону", 3, [T]),
        ex("Сфінкс", "intermediate", "Лежачи на животі, обіприся на передпліччя й м'яко підніми груди, затримайся на 10 секунд", "2", "5", 2, [P, T]),
        ex("Супермен", "intermediate", "Лежачи на животі, одночасно підійми руки й ноги, затримайся на 2-3 секунди", "3", "10", 3, [P, T]),
    ]},
    "chest": {"label": "груди", "aliases": ["груд"], "exercises": [
        ex("Віджимання від стіни", "beginner", "Стань на крок від стіни, долоні на рівні плечей, повільно згинай і розгинай руки", "2-3", "10-12", 3),
        ex("Стискання долонь на рівні грудей", "beginner", "Сидячи рівно, з'єднай долоні на рівні грудей і стискай їх 10 секунд", "2", "5", 1.5),
        ex("Віджимання від столу", "beginner", "Упрись руками в край стійкого столу, тіло рівне, опускай груди до краю", "2-3", "8-10", 3, [T]),
        ex("Віджимання з колін", "beginner", "Коліна на підлозі, тіло рівне від колін до голови, опускай груди до підлоги", "2-3", "8-10", 3, [T]),
        ex("Віджимання від підлоги", "intermediate", "Руки на ширині плечей, тіло пряме, опускайся до торкання грудьми підлоги", "3", "10-15", 3, [T]),
    ]},
    "legs": {"label": "ноги", "aliases": ["ног", "ніг", "стегн", "сідниц"], "exercises": [
        ex("Присідання на стілець", "beginner", "Повільно сідай на стілець і вставай без допомоги рук, спина пряма", "2-3", "10", 3),
        ex("Підйоми на носки біля стіни", "beginner", "Тримаючись за стіну, підіймайся на носки й повільно опускайся", "2", "15", 2),
        ex("Відведення ноги вбік з опорою", "beginner", "Тримаючись за спинку стільця, плавно відводь пряму ногу вбік", "2", "10 на кожну ногу", 2.5),
        ex("Присідання", "beginner", "Стопи на ширині плечей, спина пряма, присідай до паралелі стегон з підлогою", "3", "12-15", 3, [T]),
        ex("Сидіння біля стіни", "intermediate", "Спиною до стіни, сповзи вниз до кута 90 градусів у колінах і тримай 20-30 секунд", "3", "20-30 секунд", 2, [T]),
        ex("Випади назад з опорою", "intermediate", "Тримаючись за опору, зроби крок назад і м'яко опусти коліно до підлоги", "3", "8 на кожну ногу", 3, [T]),
    ]},
    "core": {"label": "прес", "aliases": ["прес", "живіт", "живот"], "exercises": [
        ex("Планка біля стіни", "beginner", "Упрись передпліччями в стіну, тіло рівне під кутом, тримай 20 секунд", "3", "20 секунд", 2, [T]),
        ex("Мертвий жук", "beginner", "Лежачи на спині, руки й ноги вгору, по черзі опускай протилежні руку й ногу, поперек притиснутий", "2", "8 на кожну сторону", 3, [T]),
        ex("Планка з колін", "beginner", "Упор на передпліччя й коліна, тіло рівне, тримай 15-20 секунд", "3", "15-20 секунд", 2, [T]),
        ex("Бічна планка з колін", "beginner", "Лежачи на боці, обіприся на передпліччя й коліно, підніми таз і тримай 15 секунд", "2", "15 секунд на кожен бік", 2, [T]),
        ex("Планка", "intermediate", "Упор на передпліччя й носки, тіло рівне від голови до п'ят, тримай 20-30 секунд", "3", "20-30 секунд", 2, [T]),
        ex("Скручування", "intermediate", "Лежачи на спині, ноги зігнуті, підіймай плечі від підлоги", "3", "15", 3, [P, S, T]),
    ]},
    "neck_shoulders": {"label": "шия і плечі", "aliases": ["шия", "шию", "шиї", "шиєю", "плеч", "лопат"], "exercises": [
        ex("Втягування підборіддя", "beginner", "Сидячи рівно, м'яко відведи підборіддя назад, ніби робиш подвійне підборіддя, затримайся на 3 секунди", "2", "10", 1.5),
        ex("Зведення лопаток сидячи", "beginner", "Сидячи рівно, зведи лопатки разом, затримайся на 3 секунди, розслаб", "2", "10", 1.5),
        ex("Підйоми плечей", "beginner", "Повільно підніми плечі до вух, затримайся на 2 секунди й опусти", "2", "10", 1.5),
        ex("Ангели біля стіни", "beginner", "Спиною до стіни, руки зігнуті й притиснуті, повільно ковзай ними вгору й униз", "2", "10", 2),
        ex("Нахили голови до плеча", "beginner", "Повільно нахили голову до плеча, утримуй 15 секунд, без ривків", "2", "по 15 секунд у кожен бік", 1.5, [T]),
        ex("Жим рук угору з пляшками води", "intermediate", "Сидячи, тримай пляшки на рівні плечей і плавно піднімай руки вгору", "3", "10", 2.5, [S, T]),
    ]},
    "arms": {"label": "руки", "aliases": ["рук", "біцепс", "трицепс"], "exercises": [
        ex("Згинання рук з пляшками води", "beginner", "Стоячи чи сидячи, лікті притиснуті до тулуба, згинай руки з пляшками до плечей", "2-3", "12", 2.5),
        ex("Молоток з пляшками води", "beginner", "Тримай пляшки вертикально, як молоток, і згинай руки до плечей", "2-3", "12", 2.5),
        ex("Підйоми рук уперед з пляшками", "beginner", "Тримаючи пляшки, плавно піднімай прямі руки вперед до рівня плечей", "2", "10", 2),
        ex("Ізометричне згинання з рушником", "beginner", "Стань на середину рушника, тягни кінці вгору зігнутими руками й тримай 10 секунд", "2", "5", 1.5),
        ex("Розгинання рук з-за голови з пляшкою", "intermediate", "Сидячи, тримай пляшку двома руками за головою й розгинай руки вгору", "2-3", "10", 2.5, [S]),
    ]},
    "cooldown": {"label": "заминка", "aliases": ["заминк", "розтяжк", "розтягн"], "exercises": [
        ex("Дихання животом лежачи", "beginner", "Лежачи на спині, долоня на животі, повільно вдихай носом і видихай ротом", "1", "2 хвилини", 2),
        ex("Розтяжка грудних м'язів у дверному отворі", "beginner", "Поклади передпліччя на одвірок і м'яко подайся вперед до легкого натягу, тримай 20 секунд", "1", "20 секунд на кожен бік", 1),
        ex("Розтяжка стегна стоячи з опорою", "beginner", "Тримаючись за опору, зігни ногу й підтягни п'яту до сідниці, тримай 20 секунд", "1", "20 секунд на кожну ногу", 1.5),
        ex("Розтяжка задньої поверхні стегна на стільці", "beginner", "Сидячи на краю стільця, випрями одну ногу й з рівною спиною трохи нахилися вперед", "1", "20 секунд на кожну ногу", 1.5, [T]),
        ex("Поза дитини", "beginner", "Сядь на п'яти, витягни руки вперед і опусти груди до стегон, дихай спокійно", "1", "30 секунд", 1.5, [T]),
        ex("Коліна до грудей лежачи", "beginner", "Лежачи на спині, обійми коліна й м'яко підтягни їх до грудей, тримай 20 секунд", "1", "20 секунд", 1.5, [T]),
    ]},
}

total = sum(len(g["exercises"]) for g in EXERCISES_DB.values())
safe_back = sum(1 for g in EXERCISES_DB.values() for e in g["exercises"] if P not in e["contraindications"])
print(f"✅ База: {len(EXERCISES_DB)} груп, {total} вправ для дому, з них {safe_back} доступні при болю в попереку")

✅ База: 8 груп, 46 вправ для дому, з них 43 доступні при болю в попереку


In [5]:
# Рівні активності для формули калорій (з ДЗ4).
# Ключі англійською, "factor" — коефіцієнт для денної норми, "label" — назва для людини,
# "aliases" — основи слів, за якими код упізнає рівень у відповіді ("помірн" → "помірна", "помірною").
ACTIVITY_LEVELS = {
    "sedentary":   {"factor": 1.2,   "label": "мінімальна",   "aliases": ["мінімальн", "сидяч", "сидж"]},
    "light":       {"factor": 1.375, "label": "легка",        "aliases": ["легк"]},
    "moderate":    {"factor": 1.55,  "label": "помірна",      "aliases": ["помірн", "середн"]},
    "active":      {"factor": 1.725, "label": "висока",       "aliases": ["висок"]},
    "very_active": {"factor": 1.9,   "label": "екстремальна", "aliases": ["екстремальн"]},
}

print(f"✅ Рівні активності: {', '.join(level['label'] for level in ACTIVITY_LEVELS.values())}")

✅ Рівні активності: мінімальна, легка, помірна, висока, екстремальна
